Using LIANA ligand-receptor databases to construct the ligand-target
model
================

Following the [Construction of NicheNet’s ligand-target
model](model_construction.ipynb) notebook, we will now demonstrate how to
use ligand-receptor reactions from LIANA to build the ligand-target
model. LIANA is a framework that combines both resources and
computational tools for ligand-receptor cell-cell communication
inference (Dimitrov et al., 2022). As the NicheNet prior model is built
by integrating ligand-receptor, signaling, and gene regulatory
databases, each part can be replaced with external data sources. We will
show how the first part, the ligand-receptor database, can be replaced
with those from LIANA, and how to run the model afterward.

**Important**: Since LIANA also offers functions to calculate
ligand-receptor interactions of interest, it is also possible to use
them to select which ligands are of interest to do the ligand activity
analysis. 

In [1]:
from nichenetpy.utils import read_csv_cols, read_csv_rows
from nichenetpy.model_construction import construct_weighted_networks, construct_ligand_target_matrix, apply_hub_correction
from nichenetpy.prediction import LigandActivityPredictor

from itertools import chain, repeat
from liana.resource import (
    show_resources,
    select_resource
)

import os
import requests
import pandas as pd
import re

In [2]:
human_network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(human_network_path):
    os.makedirs(human_network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv"
):
    file_path = os.path.join(human_network_path, filename)
    if not os.path.exists(file_path):
        resource = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(resource.content)
mouse_network_path = os.path.normpath("./tutorial_files/model_construction/mouse")
if not os.path.exists(mouse_network_path):
    os.makedirs(mouse_network_path)
for filename in (
    "gr_mouse.csv",
    "lr_network_mouse.csv",
    "lr_sig_mouse.csv",
):
    file_path = os.path.join(mouse_network_path, filename)
    if not os.path.exists(file_path):
        resource = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(resource.content)

In [3]:
gr_network_human = pd.DataFrame(read_csv_cols(os.path.join(human_network_path, "gr_human.csv")))
lr_network_human = pd.DataFrame(read_csv_cols(os.path.join(human_network_path, "lr_network_human.csv")))
sig_network_human = pd.DataFrame(read_csv_cols(os.path.join(human_network_path, "lr_sig_human.csv")))
gr_network_mouse = pd.DataFrame(read_csv_cols(os.path.join(mouse_network_path, "gr_mouse.csv")))
lr_network_mouse = pd.DataFrame(read_csv_cols(os.path.join(mouse_network_path, "lr_network_mouse.csv")))
sig_network_mouse = pd.DataFrame(read_csv_cols(os.path.join(mouse_network_path, "lr_sig_mouse.csv")))
source_weights = list(zip(*read_csv_rows("./tutorial_files/model_construction/human/optimized_source_weights.csv")[1]))
source_weights = dict(zip(source_weights[0], [float(e) for e in source_weights[1]]))

To check which resources are present in LIANA, we can use the show_resources() function. These are then accessed via select_resource().

In [4]:
show_resources()

['baccin2019',
 'cellcall',
 'cellchatdb',
 'cellinker',
 'cellphonedb',
 'celltalkdb',
 'connectomedb2020',
 'consensus',
 'embrace',
 'guide2pharma',
 'hpmr',
 'icellnet',
 'italk',
 'kirouac2010',
 'lrdb',
 'mouseconsensus',
 'ramilowski2015']

Next, we will calculate how much overlap there is between the ligands and receptors in the LIANA and NicheNet databases. If the overlap between LIANA receptors and NicheNet signaling network is too low, the integration will probably not work very well. 

In [5]:
mouse_re = r".*mouse.*"
n_ligands = []
n_receptors = []
n_ligands_overlap = []
n_receptors_overlap_lr = []
n_receptors_overlap_sig = []
for resource in show_resources():
    db = select_resource(resource)
    lr_network, sig_network = (
        (lr_network_human, sig_network_human)
        if re.match(mouse_re, resource) is None
        else (lr_network_mouse, sig_network_mouse)
    )
    n_ligands.append(len(set(db["ligand"])))
    n_receptors.append(len(set(db["receptor"])))
    n_ligands_overlap.append(len(set(db["ligand"]).intersection(lr_network["from"])))
    n_receptors_overlap_lr.append(len(set(db["receptor"]).intersection(lr_network["to"])))
    n_receptors_overlap_sig.append(len(set(db["receptor"]).intersection(sig_network["to"])))
overlap_df = pd.DataFrame({
    "n_ligands": n_ligands,
    "n_receptors": n_receptors,
    "n_ligands_overlap": n_ligands_overlap,
    "n_receptors_overlap_lr": n_receptors_overlap_lr,
    "n_receptors_overlap_sig": n_receptors_overlap_sig
}, index=show_resources())
overlap_df["frac_ligands_overlap"] = overlap_df["n_ligands_overlap"] / overlap_df["n_ligands"]
overlap_df["frac_ligands_overlap_lr"] = overlap_df["n_receptors_overlap_lr"] / overlap_df["n_receptors"]
overlap_df["frac_ligands_overlap_sig"] = overlap_df["n_receptors_overlap_sig"] / overlap_df["n_receptors"]
overlap_df

,n_ligands,n_receptors,n_ligands_overlap,n_receptors_overlap_lr,n_receptors_overlap_sig,frac_ligands_overlap,frac_ligands_overlap_lr,frac_ligands_overlap_sig
baccin2019,647,616,541,520,540,0.836167,0.844156,0.876623
cellcall,276,219,272,193,194,0.985507,0.881279,0.885845
cellchatdb,529,490,504,329,346,0.952741,0.671429,0.706122
cellinker,1218,1105,880,788,867,0.722496,0.713122,0.784615
cellphonedb,500,451,471,317,337,0.942000,0.702882,0.747228
celltalkdb,811,780,742,683,767,0.914920,0.875641,0.983333
connectomedb2020,812,681,769,640,668,0.947044,0.939794,0.980910
consensus,1036,1059,917,759,878,0.885135,0.716714,0.829084
embrace,462,473,435,436,470,0.941558,0.921776,0.993658
guide2pharma,293,243,289,242,243,0.986348,0.995885,1.000000


On average, ~90% of the ligands and receptors of LIANA databases are in the NicheNet LR network (frac_ligands_overlap, frac_receptors_overlap_lr), and almost all of the receptors in LIANA databases are present in the NicheNet signaling network (frac_receptors_overlap_sig). When using the “Consensus” database of LIANA, there are ~100 ligands that are not present in NicheNet; in contrast, there are 303 ligands in NicheNet that are not present in the LIANA consensus database.

To build the ligand-target model, we can use a very similar code to the Construction of NicheNet’s ligand-target model vignette. Users can choose between replacing the NicheNet LR database entirely with LIANA’s (replace_nichenet_lr = TRUE), or just adding the LIANA database as an additional data source, which may contain a lot of redundant information.

In [6]:
replace_nichenet_lr = True
liana_db = select_resource("Consensus")
liana_db.rename(columns={
    "ligand": "from",
    "receptor": "to"
}, inplace=True)
liana_db["source"] = list(repeat("liana", len(liana_db)))
if not replace_nichenet_lr:
    liana_db = pd.concat((lr_network_human, liana_db))
source_weights["liana"] = 1
weighted_networks = construct_weighted_networks(
    liana_db,
    sig_network_human,
    gr_network_human,
    source_weights
)
weighted_networks["lr_sig"] = apply_hub_correction(weighted_networks["lr_sig"], hub=0.115)
weighted_networks["gr"] = apply_hub_correction(weighted_networks["gr"], hub=0.0803)
ligands = [["TNF"], ["TNF", "IL6"]]
predictor = LigandActivityPredictor(
    *construct_ligand_target_matrix(
        weighted_networks,
        lr_network,
        ligands,
        damping_factor=0.789,
        ltf_cutoff=0.926
    )
)

## References

Dimitrov, D., Türei, D., Garrido-Rodriguez M., Burmedi P.L., Nagai,
J.S., Boys, C., Flores, R.O.R., Kim, H., Szalai, B., Costa, I.G.,
Valdeolivas, A., Dugourd, A. and Saez-Rodriguez, J. Comparison of
methods and resources for cell-cell communication inference from
single-cell RNA-Seq data. Nat Commun 13, 3224 (2022).
<https://doi.org/10.1038/s41467-022-30755-0>